# 08 - Points by Neighborhood

For a target neighborhood, builds a hex grid and counts business openings and closings per hex cell per year. Loads the full openings/closings dataset and SF neighborhood boundaries, spatially joins points to neighborhoods, then bins events into a 100m hex grid clipped to the neighborhood boundary. Exports the hex-aggregated GeoDataFrame as a parquet for use in map visualizations.

<a href="https://colab.research.google.com/github/seanflannelly/SF-Businesses-Openings-Closings/blob/main/notebooks/processing/05_points_by_neighborhood.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import requests, json
from io import BytesIO
from shapely.geometry import Polygon

In [ ]:
url = "https://raw.githubusercontent.com/seanflannelly/SF-Businesses-Openings-Closings/main/data/processed/ALL_openings_closings.parquet"

response = requests.get(url)
response.raise_for_status()

gdf_sf = gpd.read_parquet(BytesIO(response.content))

# Quick crs check
if gdf_sf.crs is None:
    gdf_sf = gdf_sf.set_crs(epsg=4326)

gdf_sf.head()

In [ ]:
# Read neighborhood boundaries

neighs_url = 'https://raw.githubusercontent.com/seanflannelly/SF-Businesses-Openings-Closings/main/data/raw/sf_neighborhoods.geojson'
sf_neighs = gpd.read_file(neighs_url)

if sf_neighs.crs is None:
    sf_neighs = sf_neighs.set_crs(epsg=4326)

sf_neighs = sf_neighs.to_crs(epsg=4326)

print(sf_neighs.columns)
print(gdf_sf.columns)

In [ ]:
# Standardize neighborhood boundary column

sf_neighs = sf_neighs.rename(columns={'nhood': 'neighborhood'})

# Drop old neighborhood column from points so spatial join is clean
gdf_sf = gdf_sf.drop(columns=["neighborhood", "nhood", "index_right"], errors="ignore")

# Spatial join
gdf_sf_neigh = gpd.sjoin(
    gdf_sf,
    sf_neighs[["neighborhood", "geometry"]],
    how="left",
    predicate="within"
)

print(gdf_sf_neigh["neighborhood"].value_counts().head(10))

In [ ]:
# Create opening and closing event points


gdf_sf_neigh["location_start_date"] = pd.to_datetime(
    gdf_sf_neigh["location_start_date"],
    errors="coerce"
)

gdf_sf_neigh["location_end_date"] = pd.to_datetime(
    gdf_sf_neigh["location_end_date"],
    errors="coerce"
)

open_points = gdf_sf_neigh[
    (gdf_sf_neigh["location_start_date"] >= "2019-01-01") &
    (gdf_sf_neigh["location_start_date"] < "2026-01-01")
].copy()

close_points = gdf_sf_neigh[
    (gdf_sf_neigh["location_end_date"] >= "2019-01-01") &
    (gdf_sf_neigh["location_end_date"] < "2026-01-01")
].copy()

open_points["status"] = "Opening"
close_points["status"] = "Closing"

open_points["event_year"] = open_points["location_start_date"].dt.year
close_points["event_year"] = close_points["location_end_date"].dt.year

In [ ]:
# Choose neighborhood (change for each target neigh)

target_neighborhood = "Financial District"

if target_neighborhood not in gdf_sf_neigh["neighborhood"].dropna().unique():
    target_neighborhood = (
        close_points
        .groupby("neighborhood")
        .size()
        .sort_values(ascending=False)
        .index[0]
    )

print("Mapping neighborhood:", target_neighborhood)

point_events = pd.concat([
    open_points[open_points["neighborhood"] == target_neighborhood],
    close_points[close_points["neighborhood"] == target_neighborhood]
], ignore_index=True)

point_events = point_events[point_events.geometry.notna()].copy()
point_events = gpd.GeoDataFrame(point_events, geometry="geometry", crs=gdf_sf_neigh.crs).to_crs(epsg=4326)

point_events["lon"] = point_events.geometry.x
point_events["lat"] = point_events.geometry.y

top_boundary = sf_neighs[
    sf_neighs["neighborhood"] == target_neighborhood
].to_crs(epsg=4326)

In [ ]:
# Build hex grid inside neighborhood boundary

points_proj = point_events.to_crs(epsg=7131)
boundary_proj = top_boundary.to_crs(epsg=7131)

hex_size = 100

minx, miny, maxx, maxy = boundary_proj.total_bounds

hexagons = []
x = minx
col = 0

while x < maxx:
    y = miny

    if col % 2 == 1:
        y += hex_size * np.sqrt(3) / 2

    while y < maxy:
        hexagon = Polygon([
            (x + hex_size * np.cos(angle), y + hex_size * np.sin(angle))
            for angle in np.linspace(0, 2 * np.pi, 7)[:-1]
        ])
        hexagons.append(hexagon)
        y += hex_size * np.sqrt(3)

    x += hex_size * 1.5
    col += 1

hex_grid = gpd.GeoDataFrame(geometry=hexagons, crs=boundary_proj.crs)

hex_grid = gpd.overlay(
    hex_grid,
    boundary_proj[["neighborhood", "geometry"]],
    how="intersection"
)

hex_grid["hex_id"] = range(len(hex_grid))

In [ ]:
# Count points by hex, year, status

points_proj = points_proj.drop(columns=["index_right"], errors="ignore")

points_with_hex = gpd.sjoin(
    points_proj,
    hex_grid[["hex_id", "geometry"]],
    how="inner",
    predicate="within"
)

hex_counts = (
    points_with_hex
    .groupby(["hex_id", "event_year", "status"])
    .size()
    .reset_index(name="count")
)

hex_counts_wide = (
    hex_counts
    .pivot_table(
        index=["hex_id", "event_year"],
        columns="status",
        values="count",
        fill_value=0
    )
    .reset_index()
)

if "Opening" not in hex_counts_wide.columns:
    hex_counts_wide["Opening"] = 0

if "Closing" not in hex_counts_wide.columns:
    hex_counts_wide["Closing"] = 0

years = list(range(2016, 2026))

hex_years = (
    pd.MultiIndex.from_product(
        [hex_grid["hex_id"], years],
        names=["hex_id", "event_year"]
    )
    .to_frame(index=False)
)

hex_years = hex_years.merge(
    hex_counts_wide,
    on=["hex_id", "event_year"],
    how="left"
)

hex_years[["Opening", "Closing"]] = hex_years[["Opening", "Closing"]].fillna(0)

hex_plot = hex_grid.merge(hex_years, on="hex_id", how="left")
hex_plot = hex_plot.to_crs(epsg=4326)

geojson_hex = json.loads(hex_plot.to_json())

max_opening = max(hex_plot["Opening"].max(), 1)
max_closing = max(hex_plot["Closing"].max(), 1)

In [ ]:
# Create output folder
import os
from google.colab import files

output_dir = "/content/CYPLAN255-Final-Project/data/processed"
os.makedirs(output_dir, exist_ok=True)

# Export the GeoDataFrame
hex_plot.to_parquet(
    f"{output_dir}/fidi_hex_open_close.parquet"
)

# Download the file
files.download(
    f"{output_dir}/fidi_hex_open_close.parquet"
)